# Boltz 3D Affinity API - Usage Guide

This notebook demonstrates how to use the Boltz2 async API to predict protein-ligand binding affinities.

**Workflow overview:**
1. Initialize the API client
2. Check API health and quota
3. *(Optional)* Upload MSA files for improved accuracy
4. *(Optional)* Clean protein PDB files via the API
5. Prepare protein and ligand inputs
6. Submit a prediction job
7. Monitor progress and retrieve results

## 1. Setup

In [ ]:
import os
import glob
import pandas as pd

from boltz_api_client import (
    AsyncApiClient,
    ProteinInput,
    LigandInput,
    load_pdb_file,
    load_molblock_file,
)

In [ ]:
API_URL = os.getenv("API_URL", "BOLTZ_API_URL")
API_KEY = os.getenv("API_KEY", "YOUR_API_KEY")

client = AsyncApiClient(API_URL, api_key=API_KEY)

## 2. Health Check & Quota

In [ ]:
health = client.health_check()
print(f"API Status: {health['status']}")

In [ ]:
quota = client.get_quota()
print(f"Quota: {quota.quota_used}/{quota.quota_max} used, {quota.quota_remaining} remaining")

## 3. Upload MSA Files

MSA (Multiple Sequence Alignment) files improve prediction accuracy. Each protein chain requires its own MSA file (CSV format). If you don't have MSA files, you can skip this step — predictions will still run, but with lower accuracy.

MSA file should be a csv file with two columns: key, sequence. `key` is for numbering the sequences from 0 to n, `sequence` contains the msa sequence.

The upload is a two-step process handled by `upload_msa()`:
1. Request a presigned S3 upload URL from the API
2. Upload the file directly to S3

The returned filename is then passed to `ProteinInput.msa_filenames`.

In [ ]:
# Upload a single MSA file
msa_filename_1 = client.upload_msa("msa_openfe/charge_annihilation__cdk2_A.csv")
msa_filename_2 = client.upload_msa("msa_openfe/charge_annihilation__cdk2_B.csv")
print(f"MSA key: {msa_filename_1}, {msa_filename_2}")

In [ ]:
# Or upload multiple MSA files at once
msa_files = glob.glob("msa_openfe/*.csv")
msa_keys = {}
for msa_file in msa_files:
    key = client.upload_msa(msa_file)
    msa_keys[os.path.basename(msa_file)] = key

print(f"Uploaded {len(msa_keys)} MSA files")

## 4. Clean Protein PDB *(Optional)*

The API provides a `/clean` endpoint that normalises a PDB file:
- Normalise atom and residue names to standard PDB conventions
- Remove terminal cap residues (ACE / NME)
- Remove all non residues molecules (like water, ions, cofactors, etc.)
- Remove all non standard residues.
- Resolve alternate atom locations — only the highest-occupancy conformation is kept
- Renumber residues and atoms sequentially from 1

In [ ]:
protein_tag = "charge_annihilation__cdk2"
protein_file = f"data/openfe_benchmark/{protein_tag}/protein.pdb"
protein_clean_file = f"data/openfe_benchmark/{protein_tag}/protein_clean.pdb"

raw_pdb = load_pdb_file(protein_file)
clean_pdb = client.clean_protein(raw_pdb)

with open(protein_clean_file, "w") as f:
    f.write(clean_pdb)

print("Protein cleaned and saved.")

## 5. Prepare Inputs

A prediction job requires:
- **One protein** (`ProteinInput`) composed of:
    - protein name (avoid spaces and dots in the protein name to pass validity checks)
    - clean protein PDB block (clean your protein in step 4 to pass protein validity checks) 
    - optional MSA filenames (one msa filename per chain, ensure your msas are uploaded in step 3 if you want to use them, otherwise use an empty list)
- **One or more ligands** (`LigandInput`): each with a name (avoid spaces and dots in the ligand name to pass validity checks) and a molblock (SDF/MOL format)

In [ ]:
protein = ProteinInput(
    name="cdk2",
    pdb_block=load_pdb_file(protein_clean_file),
    msa_filenames=[msa_filename_1, msa_filename_2],  # one MSA filename per chain, or [] to skip.
)
print(f"Protein: {protein.name}")

In [ ]:
ligand_files = glob.glob(f"data/openfe_benchmark/{protein_tag}/ligands/*.sdf")

ligands = []
for ligand_file in ligand_files:
    molblock = load_molblock_file(ligand_file)
    name = os.path.basename(ligand_file).replace(".sdf", "")
    ligands.append(LigandInput(name=name, molblock=molblock))

print(f"Loaded {len(ligands)} ligands")

## 6. Submit a Prediction Job

There are two ways to submit a job:

### Option A: Direct API submission (`create_request`)
The payload is sent directly via the API. Simple and convenient.

### Option B: S3 payload upload (`upload_payload`)
The payload is uploaded to S3 via a presigned URL, then picked up automatically. Better for large payloads (many ligands or large PDB files) as it avoids API size limits.

In [ ]:
# Option A: Direct submission
response = client.create_request(
    job_name=f"{protein_tag}-prediction-job",
    protein=protein,
    ligands=ligands,
)

parent_request_id = response["parent_request_id"]
print(f"Job submitted. parent_request_id: {parent_request_id}")
print(f"Total ligands: {response['total_ligands']}")

In [ ]:
# Option B: S3 upload (for large payloads)
parent_request_id = client.upload_payload(
    job_name=f"{protein_tag}-prediction-job",
    protein=protein,
    ligands=ligands,
)

## 7. Monitor Progress

You can either poll manually or use the built-in `wait_for_completion()` method.

In [ ]:
# Manual status check
status = client.get_request_status(parent_request_id)
summary = status.status_summary

print(f"Status: {status.status.value}")
print(f"Total:     {summary.total}")
print(f"Pending:   {summary.pending}")
print(f"Starting:   {summary.starting}")
print(f"Processing:   {summary.processing}")
print(f"Completed: {summary.completed} ({summary.completed_percent:.1f}%)")
print(f"Failed:    {summary.failed} ({summary.failed_percent:.1f}%)")

In [ ]:
# Automatic polling with progress callback
def show_progress(request):
    s = request.status_summary
    print(f"Pending:   {s.pending}, Starting:   {s.starting}, Processing:   {s.processing}, Completed: {s.completed}, Failed:    {s.failed}.")
    print(f"  {s.completed}/{s.total} completed ({s.completed_percent:.1f}%)")

final_status = client.wait_for_completion(
    parent_request_id,
    check_interval=10,
    progress_callback=show_progress,
)

print(f"\nDone! Final status: {final_status.status.value}")

## 8. Retrieve Results

Once the job is complete, retrieve the predictions. Each ligand result contains:
- `affinity_probability_binary`: probability of binding (0-1)
- `affinity_pred_value`: predicted binding free energy (kcal/mol)
- `affinity_pIC50_pred_value`: predicted pIC50 value

In [ ]:
results = client.get_results(parent_request_id)

In [ ]:
rows = []
for ligand_name, result in results.items():
    row = {"ligand_id": ligand_name, "status": result["status"]}
    if result["status"] == "COMPLETED":
        row.update(
            result["data"]
        )
    elif result["status"] == "FAILED":
        row["error"] = result["error"]
    rows.append(row)

df_results = pd.DataFrame(rows)
df_results.head()


In [ ]:
df_results.to_csv("prediction_results.csv", index=False)
print(f"Saved {len(df_results)} results to prediction_results.csv")